In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from gnninterpreter import *

In [3]:
import torch
from tqdm.auto import trange

# Mutag

In [4]:
mutag = MUTAGDataset(seed=12345)
mutag_train, mutag_val = mutag.train_test_split(k=10)
mutag_model = GCNClassifier(node_features=len(mutag.NODE_CLS),
                                num_classes=len(mutag.GRAPH_CLS),
                                hidden_channels=64,
                                num_layers=3)

In [5]:
def explanation_graph(cls_idx,cls_embeds_wt,cls_theta_wt, cls_budget):
    trainer[cls_idx] = Trainer(
        sampler=(s := GraphSampler(
            max_nodes=6,
            num_node_cls=len(mutag.NODE_CLS),
            temperature=0.15,
            learn_node_feat=True
        )),
        discriminator=mutag_model,
        criterion=WeightedCriterion([
            dict(key="logits", criterion=ClassScoreCriterion(class_idx=cls_idx, mode='maximize'), weight=1),
            dict(key="embeds", criterion=EmbeddingCriterion(target_embedding=mean_embeds[cls_idx]), weight=cls_embeds_wt),
            dict(key="logits", criterion=MeanPenalty(), weight=0),
            dict(key="omega", criterion=NormPenalty(order=1), weight=1),
            dict(key="omega", criterion=NormPenalty(order=2), weight=1),
            dict(key="xi", criterion=NormPenalty(order=1), weight=0),
            dict(key="xi", criterion=NormPenalty(order=2), weight=0),
            # dict(key="eta", criterion=NormPenalty(order=1), weight=0),
            # dict(key="eta", criterion=NormPenalty(order=2), weight=0),
            dict(key="theta_pairs", criterion=KLDivergencePenalty(binary=True), weight=cls_theta_wt),
        ]),
        optimizer=(o := torch.optim.SGD(s.parameters(), lr=1)),
        scheduler=torch.optim.lr_scheduler.ExponentialLR(o, gamma=1),
        dataset=mutag,
        budget_penalty=BudgetPenalty(budget=cls_budget, order=2, beta=1),
    )
    trainer[cls_idx].train(
        iterations=2000,
        target_probs={cls_idx: (0.9, 1.0)},
        target_size=30,
        w_budget_init=0.5,
        w_budget_inc=1.1,
        w_budget_dec=0.95,
        k_samples=32
    )

    return trainer[cls_idx].evaluate(threshold=0.5, show=True)

In [ ]:
expln_graphs_list = []
for i in range(10):

    for epoch in trange(128):
        train_loss = mutag_train.model_fit(mutag_model, lr=0.001)
        train_metrics = mutag_train.model_evaluate(mutag_model)
        val_metrics = mutag_val.model_evaluate(mutag_model)

    dataset_list_gt = mutag.split_by_class()
    dataset_list_pred = mutag.split_by_pred(mutag_model)
    
    mean_embeds = [d.model_transform(mutag_model, key="embeds").mean(dim=0) for d in dataset_list_gt]
    trainer = {}
    sampler = {}

    expln_cls1 = explanation_graph(1, 50, 4, 10)
    #expln1 = trainer_cls1.evaluate(threshold=0.5, show=True)

    expln_cls0 = explanation_graph(0, 10, 5, 15)
    #expln0 = trainer_cls0.evaluate(threshold=0.5, show=True)

    expln_graphs_list.append([expln_cls0,expln_cls1])
    
            

In [7]:
#expln_graphs_list

In [9]:
#Fidelity
import os
import networkx as nx
import numpy as np
motifs_path = '../motifs_real/motif_mutag/'
files_motif = os.listdir(motifs_path)

for index_m, file_m in enumerate(files_motif):
    filepath_m = os.path.join(motifs_path, file_m)
    #print(filepath_m)


def get_avg_fidelity(graph_list):
    class_avg_fidelity = []
    for expln_graph in graph_list:
        #expln_graph = torch_geometric.utils.to_networkx(g, to_undirected=True)
        #expln_graph = nx.from_numpy_array(A)
        fid_score_list = []
        for index_m, file_m in enumerate(files_motif):
            filepath_m = os.path.join(motifs_path, file_m)

            motif_graph = nx.read_gexf(filepath_m)

            GM = nx.algorithms.isomorphism.GraphMatcher(expln_graph, motif_graph)
            x = 1 if GM.subgraph_is_isomorphic() else 0
            fid_score_list.append(x)

        class_avg_fidelity.append(np.mean(fid_score_list))

    return np.mean(class_avg_fidelity)

avg_fidelity_list = []

for i in range(0,10):
    avg_fidelity = get_avg_fidelity(expln_graphs_list[i])

    print('Run'+str(i),avg_fidelity)
    avg_fidelity_list.append(avg_fidelity)
print(np.mean(avg_fidelity_list))

Run0 0.18085106382978722
Run1 0.17819148936170212
Run2 0.10106382978723404
Run3 0.10904255319148935
Run4 0.11968085106382978
Run5 0.11968085106382978
Run6 0.0851063829787234
Run7 0.09308510638297872
Run8 0.0851063829787234
Run9 0.031914893617021274
0.1103723404255319
